In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import numpy as np
from sklearn.preprocessing import LabelEncoder
from transformers import BertModel, BertTokenizer

### Будем тренировать на GPU

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device {device}")

Using device cuda


### Загрузим датасет: https://www.kaggle.com/datasets/chadgostopp/recsys-challenge-2015
Для загрузки нужные файлы были предварительно перенесены на Google Disk. Теперь их можно скачать с помощью ID и библиотеки gdown:

In [ ]:
import gdown

### Загружаем clicks

In [ ]:
file_id = '1dvK5w-klF3flPf3zjyU4fOu3rvAxwBVF'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'yoochoose-clicks.dat'
gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1dvK5w-klF3flPf3zjyU4fOu3rvAxwBVF
From (redirected): https://drive.google.com/uc?id=1dvK5w-klF3flPf3zjyU4fOu3rvAxwBVF&confirm=t&uuid=0d3ea4d0-b164-46cc-a67d-d09bc9150c14
To: /content/yoochoose-clicks.dat
100%|██████████| 1.49G/1.49G [00:17<00:00, 84.0MB/s]


'yoochoose-clicks.dat'

### Загружаем buys

In [ ]:
file_id = '1opl983HJrvXw8onp5ID9ZL51c3424FmN'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'yoochoose-buys.dat'
gdown.download(url, output, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1opl983HJrvXw8onp5ID9ZL51c3424FmN
To: /content/yoochoose-buys.dat
100%|██████████| 55.6M/55.6M [00:00<00:00, 209MB/s]


'yoochoose-buys.dat'

#### Датасет достаточно большой. Будем использовать несколько первых тысяч элементов из clicks, и соответствующие им из buys

In [ ]:
clicks = pd.read_csv('yoochoose-clicks.dat', header=None, names=['SessionID', 'Timestamp', 'ItemID', 'Category'], nrows=1_000)

In [ ]:
clicks.head()

,SessionID,Timestamp,ItemID,Category
0,1,2014-04-07T10:51:09.277Z,214536502,0
1,1,2014-04-07T10:54:09.868Z,214536500,0
2,1,2014-04-07T10:54:46.998Z,214536506,0
3,1,2014-04-07T10:57:00.306Z,214577561,0
4,2,2014-04-07T13:56:37.614Z,214662742,0


In [ ]:
buys = pd.read_csv('yoochoose-buys.dat', header=None, names=['SessionID', 'Timestamp', 'ItemID', 'Price', 'Quantity'])

In [ ]:
buys.head()

,SessionID,Timestamp,ItemID,Price,Quantity
0,420374,2014-04-06T18:44:58.314Z,214537888,12462,1
1,420374,2014-04-06T18:44:58.325Z,214537850,10471,1
2,281626,2014-04-06T09:40:13.032Z,214535653,1883,1
3,420368,2014-04-04T06:13:28.848Z,214530572,6073,1
4,420368,2014-04-04T06:13:28.858Z,214835025,2617,1


In [ ]:
# Извлечение уникальных идентификаторов сессий из урезанного набора clicks
session_ids_in_clicks = clicks['SessionID'].unique()

# Фильтрация buys, оставляем только те строки, где SessionID есть в session_ids_in_clicks
filtered_buys = buys[buys['SessionID'].isin(session_ids_in_clicks)]

# Проверка количества строк в отфильтрованном наборе buys
print(f"Количество строк в отфильтрованном наборе buys: {len(filtered_buys)}")

# Вывод нескольких строк для проверки
filtered_buys.head()

Количество строк в отфильтрованном наборе buys: 38


,SessionID,Timestamp,ItemID,Price,Quantity
10,11,2014-04-03T11:04:11.417Z,214821371,1046,1
11,11,2014-04-03T11:04:18.097Z,214821371,1046,1
12,12,2014-04-02T10:42:17.227Z,214717867,1778,4
23,21,2014-04-07T09:24:18.307Z,214548744,3141,1
24,21,2014-04-07T09:24:18.360Z,214838503,18745,1


In [ ]:
buys = filtered_buys.copy()

In [ ]:
# Преобразование Timestamp (используем временные метки для модели)
clicks['Timestamp'] = pd.to_datetime(clicks['Timestamp'])
buys['Timestamp'] = pd.to_datetime(buys['Timestamp'])

# Объединение уникальных значений ItemID из обоих датафреймов
all_item_ids = pd.concat([clicks['ItemID'], buys['ItemID']]).unique()

# Обучение LabelEncoder на объединенных значениях
item_encoder = LabelEncoder()
item_encoder.fit(all_item_ids)

# Преобразование ItemID в обоих датафреймах
clicks['ItemID'] = item_encoder.transform(clicks['ItemID'])
buys['ItemID'] = item_encoder.transform(buys['ItemID'])

# Добавление информации о том, завершилась ли сессия покупкой
clicks['IsBuy'] = clicks['SessionID'].isin(buys['SessionID'])

# Проверка количества уникальных товаров после преобразования
vocab_size = len(item_encoder.classes_)
print(f"Размер словаря товаров: {vocab_size}")

### Подготовка датасета

In [ ]:
BATCH_SIZE = 32
MAX_LEN = 50

class YooChooseDatasetWithBuys(Dataset):
    def __init__(self, data, max_len=MAX_LEN):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        session_items = self.data[self.data['SessionID'] == row['SessionID']]['ItemID'].values

        # Обрезаем или дополняем последовательности до MAX_LEN
        session_items = session_items[:self.max_len]
        session_items = np.pad(session_items, (0, self.max_len - len(session_items)), 'constant')  # Дополняем нулями

        # Нормализация временных меток (от 0 до 1)
        session_times = np.linspace(0, 1, len(session_items))

        # Флаг покупки
        is_buy = row['IsBuy']
        is_buys = np.array([is_buy] * len(session_items))

        return torch.tensor(session_items, dtype=torch.long), torch.tensor(session_times, dtype=torch.float32), torch.tensor(is_buys, dtype=torch.float32)

# Создание объекта датасета и даталоадера
dataset = YooChooseDatasetWithBuys(clicks)
train_dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)


### Определение модели

In [ ]:
class TASA_Transformer(nn.Module):
    """
    Класс модели, использующей BERT-энкодер и трансформер-декодер для предсказания следующего элемента в последовательности.
    """

    def __init__(self, vocab_size, hidden_dim, nhead, num_layers):
        """
        Инициализация модели.

        Параметры:
        - vocab_size: Размер словаря.
        - hidden_dim: Размерность скрытого слоя.
        - nhead: Количество голов в механизме внимания.
        - num_layers: Количество слоев в декодере трансформера.
        """
        super(TASA_Transformer, self).__init__()

        """
        1. Инициализация BERT-энкодера
        Загрузите предобученную модель BERT и используйте её для кодирования входных данных
        """

        self.bert_encoder = # YOUR CODE HERE

        """
        2. Инициализация трансформер-декодера
        Создайте слой декодера трансформера с заданными параметрами и используйте его для декодирования целевой последовательности
        """

        decoder_layer = # YOUR CODE HERE
        self.transformer_decoder = # YOU CODE HERE

        """
        3. Инициализация полносвязного слоя
        Добавьте полносвязный слой, который будет преобразовывать выходные данные декодера в прогнозы следующего элемента
        """
        self.fc = # YOUR CODE HERE

    def forward(self, src_items, src_times, tgt_items, tgt_times):
        """
        Прямой проход модели.

        Параметры:
        - src_items: Входная последовательность элементов.
        - src_times: Временные метки для входной последовательности.
        - tgt_items: Целевая последовательность элементов.
        - tgt_times: Временные метки для целевой последовательности.

        Возвращает:
        - logits: Прогнозы следующего элемента в последовательности.
        """

        """
        1. Кодирование входных данных с использованием BERT
        Используйте BERT-энкодер для кодирования входных последовательностей элементов и масок"""

        src_encoded = # YOUR CODE HERE


        """
        2. Кодирование целевой последовательности с использованием BERT
        # Используйте BERT-энкодер для кодирования целевой последовательности элементов и масок"""

        tgt_encoded = # YOUR CODE HERE


        """
        3. Декодирование целевой последовательности с использованием трансформер-декодера
        Используйте трансформер-декодер для декодирования закодированной целевой последовательности
        """

        decoded_output = # YOUR CODE HERE


        """
        4. Прогнозирование следующего элемента
        Используйте полносвязный слой для преобразования выходных данных декодера в прогнозы следующего элемента
        """

        logits = # YOUR CODE HERE
        return logits


# Определение vocab_size и других параметров
vocab_size = len(item_encoder.classes_)
hidden_dim = 768
nhead = 8
num_layers = 3

# Инициализация модели
model = TASA_Transformer(vocab_size=vocab_size, hidden_dim=hidden_dim, nhead=nhead, num_layers=num_layers)
model = model.to(device)

In [ ]:
len(train_dataloader)

### Обучение модели

In [2]:
def train_model(model, train_dataloader, device, vocab_size, epochs=20, learning_rate=0.0001):
    """
    Функция для обучения модели.

    Параметры:
    - model: Модель, которую нужно обучить.
    - train_dataloader: DataLoader для обучающих данных.
    - device: Устройство (например, 'cuda' или 'cpu'), на котором будет происходить обучение.
    - vocab_size: Размер словаря.
    - epochs: Количество эпох обучения (по умолчанию 20).
    - learning_rate: Скорость обучения (по умолчанию 0.0001).

    Описание:
    1. Инициализируйте функцию потерь (CrossEntropyLoss) и оптимизатор (Adam).
    2. В цикле по эпохам:
        - Установите модель в режим обучения.
        - Инициализируйте переменную для хранения общей потери.
        - В цикле по батчам данных:
            - Переместите данные на device.
            - Обнулите градиенты оптимизатора.
            - Подготовьте целевую последовательность.
            - Сделайте прогноз с помощью модели.
            - Приведите предсказания и цели к подходящей форме.
            - Вычислите потери.
            - Обновите параметры модели с помощью обратного распространения ошибки и оптимизатора.
            - Добавьте текущую потерю к общей потере.
        - Выведите среднюю потерю за эпоху.
    """

    criterion = # YOUR CODE HERE
    optimizer = # YOUR CODE HERE

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for src_items, src_times, is_buys in train_dataloader:

            # YOUR CODE HERE

        print(f'Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(train_dataloader)}')

# Пример использования функции
# train_model(model, train_dataloader, device, vocab_size)


In [ ]:
def evaluate_model(model, dataloader, device, vocab_size):
    """
    Функция для оценки модели на тестовых данных.

    Параметры:
    - model: Модель, которую нужно оценить.
    - dataloader: DataLoader для тестовых данных.
    - device: Устройство (например, 'cuda' или 'cpu'), на котором будет происходить оценка.
    - vocab_size: Размер словаря.

    Описание:
    1. Установите модель в режим оценки.
    2. Инициализируйте переменные для хранения количества правильных предсказаний и общего количества предсказаний.
    3. В цикле по батчам данных:
        - Переместите данные на устройство.
        - Подготовьте целевую последовательность.
        - Сделайте прогноз с помощью модели.
        - Приведите предсказания и цели к подходящей форме.
        - Вычислите предсказания.
        - Обновите количество правильных предсказаний и общее количество предсказаний.
    4. Вычислите точность и выведите её.
    5. Верните точность.
    """
    model.eval()
    correct_predictions = 0
    total_predictions = 0

    with torch.no_grad():
        for batch in dataloader:
            # YOUR CODE HERE

    accuracy = correct_predictions / total_predictions
    print(f'Test Accuracy: {accuracy * 100:.2f}%')
    return accuracy

# Пример использования модели
# evaluate_model(model, train_dataloader, device, vocab_size)